# Aggravating factor sentence adjustments — percentage increase

Calculates the percentage increase in the starting-point sentence attributable to each
common aggravating factor, using the **verified** extracted features in the shared
MongoDB cache (`stage_model_analysis_verified_features.json`).

For each aggravating factor entry the percentage increase is:

```
base = sentence_after_role_months   (fallback: starting_point_months)
pct  = enhancement_months / base
```

The model applies the role adjustment first and the aggravating factors on top, so the
judge's stated enhancement sits on the sentence **after the role adjustment**.  The base
therefore uses `sentence_after_role_months` where available and falls back to
`starting_point_months` only when the post-role sentence is missing.

Factors analysed (canonical names in the data):

1. **Multiple drugs** — `Multiple drugs`
2. **Persistent offender** — `Persistent offender`
3. **On bail** — compared with **Suspended sentence** to decide whether the two are
   similar enough to combine.
4. **Refugee/Asylum** — the extracted factor is canonicalised to `Refugee claimant`.
5. **Use of minors** — a full list of every charge that involves `Use of minors`.
6. **Import / Export** — canonicalised to `Cross-border trafficking`, restricted to
   charges where the defendant role is **not set** or is **Courier / Storekeeper**
   (severe-role charges are dropped because the role adjustment already covers the
   cross-border uplift there).

Only charges with a measurable, non-inferred enhancement are included in the percentage
statistics.  The medians are computed both on all such charges and after removing
outliers (1.5 x IQR Tukey fences per factor).  The current model applies a flat **4%**
(`aggravatingAdjustment`) per aggravating factor, so the medians below show where a
per-factor percentage might be justified.


In [1]:

import numpy as np
import pandas as pd

from linear_interpolation_model import (
	flatten_documents,
	get_notebook_dir,
	load_documents,
	load_role_catalogue,
)

notebook_dir = get_notebook_dir()
documents, cache_metadata = load_documents(notebook_dir, refresh_cache=False)
trial_rows, effect_rows = flatten_documents(documents)
catalogue, role_provenance = load_role_catalogue(notebook_dir)

charge_cols = [
	"neutral_citation",
	"trial_index",
	"charge_no",
	"defendant_id",
	"role_catalogue_key",
	"starting_point_months",
	"sentence_after_role_months",
]

effects = effect_rows[effect_rows["stage"] == "aggravation"].copy()
effects = effects.merge(
	catalogue[["role_catalogue_key", "workbook_primary_role"]],
	on="role_catalogue_key",
	how="left",
)
effects = effects.merge(
	trial_rows[charge_cols].drop_duplicates("role_catalogue_key"),
	on="role_catalogue_key",
	how="left",
)

post_role_base = effects["sentence_after_role_months"].notna() & (
	effects["sentence_after_role_months"] > 0
)
effects["base_months"] = np.where(
	post_role_base,
	effects["sentence_after_role_months"],
	effects["starting_point_months"],
)
effects["pct_increase"] = effects["adjustment_months"] / effects["base_months"] * 100

print(f"Documents: {len(documents)}")
print(f"Trial rows: {len(trial_rows)}")
print(f"Aggravating effect rows (measurable enhancement): {len(effects)}")
print(
	f"Base used: sentence after role for {int(post_role_base.sum())} rows, "
	f"fallback to starting point for {int((~post_role_base).sum())} rows"
)


Documents: 2308
Trial rows: 3004
Aggravating effect rows (measurable enhancement): 995
Base used: sentence after role for 954 rows, fallback to starting point for 41 rows


In [2]:

def factor_summary(frame):
	rows = []
	for factor, group in frame.groupby("canonical_factor", sort=False):
		pooled = group["adjustment_months"].sum() / group["base_months"].sum() * 100
		rows.append({
			"factor": factor,
			"charges": len(group),
			"mean_pct": group["pct_increase"].mean(),
			"median_pct": group["pct_increase"].median(),
			"min_pct": group["pct_increase"].min(),
			"max_pct": group["pct_increase"].max(),
			"pooled_pct": pooled,
		})
	return pd.DataFrame(rows)


def iqr_outlier_mask(frame, value_col="pct_increase"):
	mask = pd.Series(False, index=frame.index)
	for factor, group in frame.groupby("canonical_factor", sort=False):
		q1 = group[value_col].quantile(0.25)
		q3 = group[value_col].quantile(0.75)
		iqr = q3 - q1
		lower = q1 - 1.5 * iqr
		upper = q3 + 1.5 * iqr
		mask.loc[group.index] = (group[value_col] < lower) | (group[value_col] > upper)
	return mask


summary_factors = [
	"Multiple drugs",
	"Persistent offender",
	"On bail",
	"Suspended sentence",
	"Refugee claimant",
	"Use of minors",
]

standard_effects = effects[effects["canonical_factor"].isin(summary_factors)]
outlier_mask = iqr_outlier_mask(standard_effects)
cleaned_standard = standard_effects[~outlier_mask]

print("Outliers removed per factor (1.5 x IQR Tukey fences):")
outlier_counts = standard_effects.assign(outlier=outlier_mask)
print(
	outlier_counts.groupby("canonical_factor")["outlier"]
	.agg(["count", "sum"])
	.rename(columns={"count": "charges", "sum": "removed"})
	.to_string()
)
print()
print(f"Total charges: {len(standard_effects)}  ->  {len(cleaned_standard)} after outlier removal")
print()

summary = factor_summary(standard_effects)
summary_clean = factor_summary(cleaned_standard)
from IPython.display import display
print("Percentage increase per aggravating factor, outliers removed "
      "(current model default is 4%):")
display(summary_clean.round(2))
print("For comparison, percentages on the full (uncleaned) data:")
display(summary.round(2))


Outliers removed per factor (1.5 x IQR Tukey fences):
                     charges  removed
canonical_factor                     
Multiple drugs           383       30
On bail                   58        2
Persistent offender      254        7
Refugee claimant          52        1
Suspended sentence         5        1
Use of minors             18        0

Total charges: 770  ->  729 after outlier removal

Percentage increase per aggravating factor, outliers removed (current model default is 4%):


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Persistent offender,247,4.40,4.00,0.0,13.64,3.84
1,Multiple drugs,353,4.01,3.85,0.0,11.76,3.36
2,Refugee claimant,51,7.73,8.00,0.0,16.67,6.03
3,On bail,56,4.37,4.41,0.0,10.71,3.91
4,Use of minors,18,6.29,5.25,0.0,20.00,5.85
5,Suspended sentence,4,2.93,3.12,0.0,5.45,1.40


For comparison, percentages on the full (uncleaned) data:


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Persistent offender,254,4.95,4.17,0.0,50.00,4.02
1,Multiple drugs,383,5.60,3.92,0.0,87.80,4.21
2,Refugee claimant,52,8.06,8.08,0.0,25.00,6.04
3,On bail,58,4.69,4.58,0.0,14.71,4.14
4,Use of minors,18,6.29,5.25,0.0,20.00,5.85
5,Suspended sentence,5,4.84,4.84,0.0,12.50,1.49


In [3]:

bail = effects[effects["canonical_factor"].isin(["On bail", "Suspended sentence"])].copy()
comparison = factor_summary(bail)
print("On bail vs Suspended sentence:")
display(comparison.round(2))
print()
print("Suspended sentence per-charge detail (all charges):")
sus_detail = [
	"neutral_citation",
	"trial_index",
	"charge_no",
	"defendant_id",
	"base_months",
	"adjustment_months",
	"pct_increase",
]
print(
	effects[effects["canonical_factor"] == "Suspended sentence"][sus_detail]
	.sort_values("pct_increase", ascending=False)
	.to_string(index=False)
)


On bail vs Suspended sentence:


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,On bail,58,4.69,4.58,0.0,14.71,4.14
1,Suspended sentence,5,4.84,4.84,0.0,12.50,1.49



Suspended sentence per-charge detail (all charges):
 neutral_citation  trial_index  charge_no  defendant_id  base_months  adjustment_months  pct_increase
 [2021] HKDC 1562            0          1             1          4.0                0.5     12.500000
 [2025] HKDC 1054            0          3             2         55.0                3.0      5.454545
 [2023] HKDC 1189            0          5             1         62.0                3.0      4.838710
  [2023] HKDC 306            0          1             1         71.0                1.0      1.408451
[2021] HKCFI 2697            2          3             2        312.0                0.0      0.000000


In [4]:

uom_trial = trial_rows[trial_rows["canonical_aggravating_factors"].map(
	lambda factors: "Use of minors" in factors
)].copy()
uom_trial = uom_trial.merge(
	catalogue[["role_catalogue_key", "workbook_primary_role"]],
	on="role_catalogue_key",
	how="left",
)
uom_effects = effects[effects["canonical_factor"] == "Use of minors"].copy()

print(
	f"Charges involving 'Use of minors': {len(uom_trial)} total, "
	f"{len(uom_effects)} with a measurable enhancement"
)
print()
print("All 'Use of minors' charges:")
detail_cols = [
	"neutral_citation",
	"trial_index",
	"charge_no",
	"defendant_id",
	"starting_point_months",
	"sentence_after_role_months",
	"workbook_primary_role",
]
display(uom_trial[detail_cols].sort_values(["neutral_citation", "trial_index"]))
print()
print("Measurable 'Use of minors' enhancements (largest first):")
print(
	uom_effects[["neutral_citation", "trial_index", "base_months", "adjustment_months", "pct_increase"]]
	.sort_values("pct_increase", ascending=False)
	.to_string(index=False)
)


Charges involving 'Use of minors': 30 total, 18 with a measurable enhancement

All 'Use of minors' charges:


,neutral_citation,trial_index,charge_no,defendant_id,starting_point_months,sentence_after_role_months,workbook_primary_role
27,[2021] HKCFI 2396,1,1,2,136.0,136.0,NaN
9,[2022] HKCFI 2559,0,1,1,162.0,162.0,NaN
20,[2022] HKCFI 2610,0,1,1,110.0,110.0,NaN
29,[2022] HKCFI 2619,0,1,1,137.0,180.0,NaN
28,[2022] HKDC 1117,0,1,1,54.0,54.0,NaN
21,[2022] HKDC 198,3,19,1,54.0,54.0,NaN
18,[2023] HKCFI 3267,0,1,1,270.0,270.0,NaN
19,[2023] HKCFI 3267,1,1,2,282.0,282.0,NaN
14,[2023] HKDC 1167,0,1,1,66.0,66.0,NaN
12,[2023] HKDC 1222,0,1,1,NaN,57.0,None



Measurable 'Use of minors' enhancements (largest first):
 neutral_citation  trial_index  base_months  adjustment_months  pct_increase
[2022] HKCFI 2619            0        180.0               36.0     20.000000
 [2022] HKDC 1117            0         54.0                6.0     11.111111
 [2025] HKDC 1428            0         57.0                6.0     10.526316
  [2024] HKDC 890            1         51.0                5.0      9.803922
  [2024] HKDC 890            2         51.0                5.0      9.803922
 [2023] HKDC 1167            0         66.0                6.0      9.090909
 [2024] HKDC 1043            0         61.0                5.0      8.196721
[2022] HKCFI 2559            0        162.0               12.0      7.407407
[2021] HKCFI 2396            1        136.0                8.0      5.882353
  [2024] HKDC 583            1         65.0                3.0      4.615385
[2025] HKCFI 1826            0        276.0               12.0      4.347826
[2025] HKCFI 1826 

In [5]:

cross_border = effects[effects["canonical_factor"] == "Cross-border trafficking"].copy()
role_is_none = cross_border["workbook_primary_role"].isna()
kept = cross_border[
	role_is_none | cross_border["workbook_primary_role"].eq("Courier / Storekeeper")
].copy()
dropped = cross_border[~role_is_none].copy()

print(f"Cross-border (Import/Export) effect rows: {len(cross_border)}")
print(f"  kept (role not set or Courier / Storekeeper): {len(kept)}")
print(f"  dropped (severe role set): {len(dropped)}")
if len(dropped):
	print()
	print("Dropped rows (role already covers the cross-border uplift):")
	print(
		dropped.groupby("workbook_primary_role")["pct_increase"]
		.agg(["count", "median"])
		.round(2)
	)
print()
cb_outlier = iqr_outlier_mask(kept)
kept_clean = kept[~cb_outlier]
print(f"Cross-border outliers removed: {int(cb_outlier.sum())} of {len(kept)}")
print()
cb_summary = factor_summary(kept)
cb_summary_clean = factor_summary(kept_clean)
print("Cross-border percentage increase, outliers removed (role not set / Courier):")
display(cb_summary_clean.round(2))
print("For comparison, percentages on the full (uncleaned) data:")
display(cb_summary.round(2))
summary = pd.concat([summary, cb_summary], ignore_index=True)
summary_clean = pd.concat([summary_clean, cb_summary_clean], ignore_index=True)
print()
print("Cross-border (role not set / Courier) per-charge detail (first 15 by size):")
print(
	kept[["neutral_citation", "trial_index", "workbook_primary_role", "base_months", "adjustment_months", "pct_increase"]]
	.sort_values("pct_increase", ascending=False)
	.head(15)
	.to_string(index=False)
)


Cross-border (Import/Export) effect rows: 149
  kept (role not set or Courier / Storekeeper): 141
  dropped (severe role set): 8

Dropped rows (role already covers the cross-border uplift):
                                 count  median
workbook_primary_role                         
Actual trafficker                    2   13.36
Manager / Organiser                  4    7.54
Operator / Financial controller      2    6.41

Cross-border outliers removed: 2 of 141

Cross-border percentage increase, outliers removed (role not set / Courier):


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Cross-border trafficking,139,5.95,5.88,0.0,13.33,5.93


For comparison, percentages on the full (uncleaned) data:


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Cross-border trafficking,141,6.16,5.88,0.0,23.08,5.95



Cross-border (role not set / Courier) per-charge detail (first 15 by size):
 neutral_citation  trial_index workbook_primary_role  base_months  adjustment_months  pct_increase
 [2024] HKDC 1937            0                   NaN         26.0                6.0     23.076923
 [2024] HKDC 2161            0                   NaN         16.0                3.0     18.750000
 [2025] HKDC 1367            0                   NaN         45.0                6.0     13.333333
[2023] HKCFI 1151            0                  None        324.0               36.0     11.111111
 [2025] HKDC 1621            0                   NaN         57.0                6.0     10.526316
   [2025] HKDC 89            2                   NaN         40.0                4.0     10.000000
[2025] HKCFI 1565            0                   NaN        153.0               15.0      9.803922
[2022] HKCFI 3441            0                   NaN        246.0               24.0      9.756098
[2021] HKCFI 3119            0  

In [6]:

def test_set(frame, per_factor=3):
	chunks = []
	for factor in summary_factors:
		group = frame[frame["canonical_factor"] == factor]
		if group.empty:
			continue
		chunks.append(group.sort_values("case_id").head(per_factor))
	return pd.concat(chunks, ignore_index=True)

test_cols = [
	"canonical_factor",
	"neutral_citation",
	"trial_index",
	"charge_no",
	"defendant_id",
	"base_months",
	"adjustment_months",
	"pct_increase",
]
test = test_set(effects)
print("Small test set per factor (for manual verification against the judgments):")
print(test[test_cols].round(2).to_string(index=False))


Small test set per factor (for manual verification against the judgments):
   canonical_factor  neutral_citation  trial_index  charge_no  defendant_id  base_months  adjustment_months  pct_increase
     Multiple drugs [2021] HKCFI 1919            1          2             1        178.0                2.0          1.12
     Multiple drugs [2021] HKCFI 1984            0          1             1         86.0                4.0          4.65
     Multiple drugs [2021] HKCFI 2074            1          2             1        156.0                3.0          1.92
Persistent offender [2021] HKCFI 1269            1          2             1        288.0                0.0          0.00
Persistent offender [2021] HKCFI 1393            0          1             1        118.0               12.0         10.17
Persistent offender [2021] HKCFI 1442            0          1             1         99.0                0.0          0.00
            On bail  [2021] HKDC 1503            0          1          

## Error rate of the predicted percentages

Each factor is predicted at its median percentage.  For every charge:

```
predicted months  = base_months * predicted_pct
error months      = predicted months - actual enhancement months
```

`mape_pct` and the `within_25/50pct` columns are computed only on charges with a
non-zero actual enhancement.  The per-factor medians are compared against the current
flat **4%** model.


In [7]:

eval_factors = summary_factors + ["Cross-border trafficking"]
eval_effects = pd.concat(
	[effects[effects["canonical_factor"].isin(summary_factors)], kept],
	ignore_index=True,
)


def evaluate_predictions(predicted_by_factor):
	rows = []
	for factor, group in eval_effects.groupby("canonical_factor", sort=False):
		fraction = predicted_by_factor[factor]
		predicted_months = group["base_months"] * fraction
		error_months = predicted_months - group["adjustment_months"]
		positive = group["adjustment_months"] > 0
		rel = (error_months.abs() / group["adjustment_months"])[positive]
		rows.append({
			"factor": factor,
			"predicted_pct": round(fraction * 100, 2),
			"charges": len(group),
			"mae_months": round(error_months.abs().mean(), 2),
			"median_abs_error_months": round(error_months.abs().median(), 2),
			"mape_pct": round(rel.mean() * 100, 1),
			"within_25pct_pct": round((rel <= 0.25).mean() * 100, 1),
			"within_50pct_pct": round((rel <= 0.50).mean() * 100, 1),
			"within_6_months_pct": round((error_months.abs() <= 6).mean() * 100, 1),
			"within_12_months_pct": round((error_months.abs() <= 12).mean() * 100, 1),
		})
	report = pd.DataFrame(rows)
	predicted_all = (
		eval_effects["base_months"] * eval_effects["canonical_factor"].map(predicted_by_factor)
	)
	error_all = predicted_all - eval_effects["adjustment_months"]
	positive_all = eval_effects["adjustment_months"] > 0
	rel_all = (error_all.abs() / eval_effects["adjustment_months"])[positive_all]
	overall = {
		"factor": "ALL",
		"predicted_pct": np.nan,
		"charges": len(eval_effects),
		"mae_months": round(error_all.abs().mean(), 2),
		"median_abs_error_months": round(error_all.abs().median(), 2),
		"mape_pct": round(rel_all.mean() * 100, 1),
		"within_25pct_pct": round((rel_all <= 0.25).mean() * 100, 1),
		"within_50pct_pct": round((rel_all <= 0.50).mean() * 100, 1),
		"within_6_months_pct": round((error_all.abs() <= 6).mean() * 100, 1),
		"within_12_months_pct": round((error_all.abs() <= 12).mean() * 100, 1),
	}
	return pd.concat([report, pd.DataFrame([overall])], ignore_index=True)


median_report = evaluate_predictions(summary.set_index("factor")["median_pct"] / 100)
clean_report = evaluate_predictions(summary_clean.set_index("factor")["median_pct"] / 100)
flat_report = evaluate_predictions(pd.Series(0.04, index=eval_factors))

print("Error rate using the predicted (median) percentage per factor:")
display(median_report)
print()
print("Error rate using the outlier-cleaned median percentages:")
display(clean_report)
print()
print("Error rate using the current flat 4% aggravating adjustment:")
display(flat_report)


Error rate using the predicted (median) percentage per factor:


,factor,predicted_pct,charges,mae_months,median_abs_error_months,mape_pct,within_25pct_pct,within_50pct_pct,within_6_months_pct,within_12_months_pct
0,Persistent offender,4.17,254,2.67,1.81,57.8,26.6,59.2,88.2,99.6
1,Multiple drugs,3.92,383,2.29,1.22,60.9,33.4,63.0,90.9,99.0
2,Refugee claimant,8.08,52,3.91,1.93,48.7,43.1,66.7,80.8,92.3
3,On bail,4.58,58,1.95,1.39,66.0,30.8,69.2,94.8,98.3
4,Use of minors,5.25,18,4.38,2.50,51.2,26.7,66.7,83.3,94.4
5,Suspended sentence,4.84,5,3.64,0.34,79.0,50.0,50.0,80.0,80.0
6,Cross-border trafficking,5.88,141,4.38,3.53,39.0,40.1,79.6,68.8,97.2
7,ALL,NaN,911,2.84,1.67,56.0,33.1,65.3,86.2,98.2



Error rate using the outlier-cleaned median percentages:


,factor,predicted_pct,charges,mae_months,median_abs_error_months,mape_pct,within_25pct_pct,within_50pct_pct,within_6_months_pct,within_12_months_pct
0,Persistent offender,4.00,254,2.64,1.68,56.3,25.7,58.7,89.4,99.6
1,Multiple drugs,3.85,383,2.27,1.19,59.9,35.1,62.7,90.9,99.0
2,Refugee claimant,8.00,52,3.86,2.00,48.2,43.1,66.7,80.8,92.3
3,On bail,4.41,58,1.92,1.25,64.0,28.8,69.2,94.8,98.3
4,Use of minors,5.25,18,4.38,2.50,51.2,26.7,66.7,83.3,94.4
5,Suspended sentence,3.12,5,2.74,1.22,68.7,0.0,50.0,80.0,100.0
6,Cross-border trafficking,5.88,141,4.38,3.53,39.0,40.1,79.6,68.8,97.2
7,ALL,NaN,911,2.81,1.65,55.0,33.3,65.1,86.5,98.4



Error rate using the current flat 4% aggravating adjustment:


,factor,predicted_pct,charges,mae_months,median_abs_error_months,mape_pct,within_25pct_pct,within_50pct_pct,within_6_months_pct,within_12_months_pct
0,Persistent offender,4.0,254,2.64,1.68,56.3,25.7,58.7,89.4,99.6
1,Multiple drugs,4.0,383,2.31,1.20,62.0,33.7,62.4,90.1,99.0
2,Refugee claimant,4.0,52,3.52,3.32,45.8,19.6,47.1,86.5,100.0
3,On bail,4.0,58,1.90,1.26,60.2,32.7,75.0,94.8,100.0
4,Use of minors,4.0,18,4.23,2.96,47.7,26.7,40.0,94.4,94.4
5,Suspended sentence,4.0,5,3.20,0.80,74.0,25.0,50.0,80.0,80.0
6,Cross-border trafficking,4.0,141,5.50,3.56,38.5,28.5,70.1,63.8,86.5
7,ALL,NaN,911,2.98,1.68,55.4,29.7,62.1,85.9,97.1


In [8]:

report_path = notebook_dir / "aggravating_factor_adjustments_analysis.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
	summary_clean.round(2).to_excel(writer, sheet_name="summary (outliers removed)", index=False)
	summary.round(2).to_excel(writer, sheet_name="summary (all)", index=False)
	comparison.round(2).to_excel(writer, sheet_name="on bail vs suspended", index=False)
	uom_trial[detail_cols].to_excel(writer, sheet_name="use of minors", index=False)
	kept.sort_values("pct_increase", ascending=False).to_excel(
		writer, sheet_name="cross-border no-role", index=False
	)
	test.round(2).to_excel(writer, sheet_name="test set", index=False)
	median_report.to_excel(writer, sheet_name="error - predicted medians", index=False)
	clean_report.to_excel(writer, sheet_name="error - cleaned medians", index=False)
	flat_report.to_excel(writer, sheet_name="error - flat 4%", index=False)
print("Wrote", report_path)


Wrote /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/aggravating_factor_adjustments_analysis.xlsx


## Reading the results

- `median_pct` is the robust central estimate for a factor; `pooled_pct` is dominated by
  large cases.
- Percentages are relative to the sentence **after the role adjustment** (falling back
  to the starting point), matching where the aggravating enhancement is applied.
- Outlier removal (1.5 x IQR Tukey fences per factor) trims extreme enhancements before
  the median is taken; the error rate then tests the cleaned median on all charges.
- Compare the medians against the current flat **4%** aggravating default.  The
  On-bail and Suspended-sentence medians are close, supporting combining the two factors.
- Small groups (e.g. Suspended sentence with 5 charges) are illustrative only.
- The Import/Export numbers cover only charges with **no role** or a **Courier /
  Storekeeper** role; severe-role cross-border is captured by the role adjustment.
